In [1]:
import os
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import LoraConfig, get_peft_model

torch.cuda.empty_cache()

# model_name = "meta-llama/Llama-3.2-1B"
model_name = "Qwen/Qwen3-0.6B"
model_save_dir = "model/downloaded"
peft_config = LoraConfig(
    r = 8,
    lora_alpha=16,
    lora_dropout=0.1,
    bias="none" ,
    target_modules=["q_proj","v_proj", "o_proj", "k_proj", "up_proj", "down_proj", "gate_proj"],
)
os.makedirs(model_save_dir, exist_ok=True)

# load the tokenizer and the model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype="auto",
    device_map="cuda",
    cache_dir=model_save_dir
)
# peft_model = get_peft_model(model, peft_config)

In [5]:
model.config.num_attention_heads
model.config.head_dim

128

In [ ]:
peft_model.config.num_attention_heads
peft_model.config.head_dim
peft_model.config.num_key_value_heads

NameError: name 'peft_model' is not defined

In [2]:
from pruning.utils import prepare_calibration_input
from data.datasets import get_loaders
train_data, val_data = get_loaders(name = 'harrison', nsamples=128, seed=42, seqlen=2048, tokenizer=tokenizer)

Token indices sequence length is longer than the specified maximum sequence length for this model (4914897 > 131072). Running this sequence through the model will result in indexing errors


In [7]:
peft_model

PeftModel(
  (base_model): LoraModel(
    (model): Qwen3ForCausalLM(
      (model): Qwen3Model(
        (embed_tokens): Embedding(151936, 1024)
        (layers): ModuleList(
          (0-27): 28 x Qwen3DecoderLayer(
            (self_attn): Qwen3Attention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=1024, out_features=2048, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.1, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=1024, out_features=8, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=8, out_features=2048, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
              (k_proj): lora.Linear(
            

In [ ]:
from pruning.WrappedGPT import WrappedGPT
from pruning.utils import find_layers, get_layers, prepare_calibration_input
import torch.nn as nn
from pruning.utils import layer_forward
def local_prune_importance(model, 
                prune_ratio,
                mask_lr,
                nsamples=128,
                seed=42,
                seqlen=2048,
                tokenizer=None,
                dataset_name = 'c4',
                reg=1e-8,
                PBS = True):
    use_cache = model.config.use_cache
    model.config.use_cache = False

    num_q_heads = model.config.num_attention_heads
    head_dim = model.config.head_dim
    num_kv_groups = model.config.num_key_value_heads

    model.eval()
    if device is None:
        device = next(model.parameters()).device

    print("loading calibration data")
    dataloader, _ = get_loaders(
        dataset_name,
        nsamples=nsamples,
        seed=seed,
        seqlen=seqlen,
        tokenizer=tokenizer,
    )
    print("dataset loading complete")

    with torch.no_grad():
        inps, outs, attention_mask, position_ids = prepare_calibration_input(model, dataloader)
    
    layers = get_layers(model)
    
    for i in range(len(layers)):
        layer = layers[i]
        subset = find_layers(layer, layers=(nn.Linear,))

        # Handle model parallel device map (optional)
        if hasattr(model, "hf_device_map") and f"model.layers.{i}" in model.hf_device_map:
            dev = model.hf_device_map[f"model.layers.{i}"]
            inps = inps.to(dev)
            outs = outs.to(dev)
            attention_mask = attention_mask.to(dev) if attention_mask is not None else None
            position_ids = position_ids.to(dev) if position_ids is not None else None
        else:
            dev = inps.device

        # Wrap linears to collect activation stats
        wrapped_layers = {name: WrappedGPT(mod, layer_id=i, layer_name=name) for name, mod in subset.items()}

        def add_batch(name):
            def _hook(mod, inp, out):
                # inp is a tuple; inp[0] is the tensor input to this Linear
                wrapped_layers[name].add_batch(inp[0].detach(), out.detach())
            return _hook

        handles = []
        for name, mod in subset.items():
            handles.append(mod.register_forward_hook(add_batch(name)))

        # Run layer forward nsamples times to populate wrapped_layers[name].scaler_row
        # IMPORTANT: Use Qwen3-correct layer forward (provides position_embeddings)
        for j in range(nsamples):
            hs = inps[j].unsqueeze(0)  # [1, seqlen, hidden]
            outs[j] = layer_forward(
                model=model,
                layer=layer,
                hidden_states=hs,
                attention_mask=attention_mask,
                position_ids=position_ids,
            ).squeeze(0)

        for h in handles:
            h.remove()

        #please edit the following part to compute correct importance score and add it to score, set mask based on the importance score
        for name in subset:
            # Access the relevant tensors for each module.
            print(f"Pruning layer {i}, param {name}")
            weight       = subset[name].base_layer.weight
            lora_A       = subset[name].lora_A.default.weight
            lora_B       = subset[name].lora_B.default.weight
            score_tensor = subset[name].score
            # X_hat is computed from the activation statistics (assumed to be feature norm)
            X_hat = torch.sqrt(wrapped_layers[name].scaler_row.reshape((1, -1))).to(mod.weight.device)
            # Save the original devices for score and mask
            orig_score_device = score_tensor.device
            orig_mask_device = subset[name].mask.device

            # Move score and mask to the current computation device (assume weight's device)
            device = weight.device
            if score_tensor.device != device:
                score_tensor = score_tensor.to(device)
            if subset[name].mask.device != device:
                subset[name].mask = subset[name].mask.to(device)
            # print(f"Module: {name}")
            # print(f"  weight shape:           {weight.shape}")
            # print(f"  lora_A shape:           {lora_A.shape}")
            # print(f"  lora_B shape:           {lora_B.shape}")
            # print(f"  X_hat shape:            {X_hat.shape}")

            # Determine the complementary effective weight.
            if name == "self_attn.q_proj":
                # Effective Q weight: shape (2048, 2048)
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                
                # Reshape Q weights into (num_q_heads, head_dim, in_features)
                effective_weight_Q = effective_weight.view(num_q_heads, head_dim, -1)  # (32, 64, 2048)
                abs_Q = torch.abs(effective_weight_Q)  # |W^Q + B^QA^Q|

                # Get corresponding KV group weights
                comp_layer = subset["self_attn.k_proj"]
                comp_weight = comp_layer.base_layer.weight + torch.matmul(
                                comp_layer.lora_B.default.weight,
                                comp_layer.lora_A.default.weight)
                
                # Reshape K weights into (num_kv_groups, head_dim, in_features)
                effective_weight_K = comp_weight.view(num_kv_groups, head_dim, -1)  # (8, 64, 2048)
                
                # Compute KV group norms (per input feature, per group)
                kv_group_norms = torch.norm(effective_weight_K, p=2, dim=1)  # (8, 2048)

                # Map Q heads to KV groups (32 Q heads → 8 KV groups)
                group_size = num_q_heads // num_kv_groups  # 4
                kv_group_idx = torch.arange(num_q_heads) // group_size  # (32,) → [0,0,0,0,1,1,...,7,7,7,7]
                
                # Expand KV norms to match Q heads
                kv_norms_expanded = kv_group_norms[kv_group_idx]  # (32, 2048)

                # Compute input column norms (precomputed ||X[:,i]||_2)
                # X_hat shape: (2048,)

                # Importance = |W^Q| * ||W^K_group|| * ||X||
                importance = (abs_Q * kv_norms_expanded.unsqueeze(1) * X_hat.unsqueeze(1))

                
                # Reshape back to original weight shape (2048, 2048)
                importance = importance.view(-1, importance.shape[-1])  # (2048, 2048)
            if name == "self_attn.k_proj":
                # Compute the effective key weight: shape (512, 2048)
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                
                k_weight_grouped = effective_weight.view(num_kv_groups, head_dim, -1)  # (8, 64, 2048)
                k_weight_abs = torch.abs(k_weight_grouped)  # |W^K + B^K A^K|

                # Get corresponding Q weights and compute group norms
                comp_layer = subset["self_attn.q_proj"]
                q_weight = comp_layer.base_layer.weight + torch.matmul(
                                comp_layer.lora_B.default.weight,
                                comp_layer.lora_A.default.weight)
                
                # Reshape Q weights into (num_kv_groups, group_size, head_dim, in_features)
                q_weight_grouped = q_weight.view(num_kv_groups, num_q_heads // num_kv_groups, head_dim, -1)  # (8, 4, 64, 2048)
                q_group_flat = q_weight_grouped.view(num_kv_groups, -1, q_weight.shape[-1])  # (8, 256, 2048)
                
                # Compute L2 norm over Q group rows (256) for each column j
                q_group_norm = torch.norm(q_group_flat, p=2, dim=1, keepdim=True)  # (8, 1, 2048)

                # Reshape X_hat_abs for broadcasting: (1, 1, 2048)
                X_hat = X_hat.view(1, 1, -1)

                # Importance = |W^K| * ||W^Q_group|| * ||X||
                importance_grouped = k_weight_abs * q_group_norm * X_hat # (8, 64, 2048)
                
                # Reshape back to original key weight shape (512, 2048)
                importance = importance_grouped.view(-1, importance_grouped.shape[-1])

            elif name == "self_attn.v_proj":
                # Compute effective v_proj weight: shape [512, 2048] → 8 heads × 64 rows
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                
                # Use self_attn.o_proj as the complementary layer.
                comp_layer = subset["self_attn.o_proj"]
                comp_weight = comp_layer.base_layer.weight + torch.matmul(
                                    comp_layer.lora_B.default.weight,
                                    comp_layer.lora_A.default.weight)
                # comp_weight has shape [2048, 2048].
                # Under attention concatenation logic, o_proj is applied to the concatenated v outputs.
                # Thus, we interpret o_proj as being divided into 8 contiguous blocks (one per v head),
                # each with 256 rows (since 8 * 256 = 2048).
                num_v_heads = num_kv_groups  
                # Determine per-head row counts:
                # For v_proj effective weight, divide its row dimension by the number of v heads.
                v_proj_head_rows = effective_weight.shape[0] // num_v_heads
                # For the complementary weight (from o_proj), divide its row dimension similarly.
                comp_head_rows = comp_weight.shape[0] // num_v_heads
                o_proj_grouped = comp_weight.view(num_v_heads, comp_head_rows, comp_weight.shape[-1])
                
                # Compute the L2 norm over each block (over the 256 rows) for every column.
                comp_norm = torch.norm(o_proj_grouped, p=2, dim=1, keepdim=True)  # shape: (8, 1, 2048)
                
                # Reshape v_proj effective weight into 8 heads of 64 rows:
                effective_weight_grouped = effective_weight.view(num_v_heads, v_proj_head_rows, effective_weight.shape[-1]) # shape: (8, 64, 2048)
                
                # Multiply elementwise:
                # - torch.abs(effective_weight_grouped): (8, 64, 2048)
                # - comp_norm: (8, 1, 2048) broadcast along the head-element dimension (64)
                # - X_hat: (1, 2048) broadcast along the head and head-element dimensions.
                importance_grouped = torch.abs(effective_weight_grouped) * comp_norm * X_hat  # shape: (8, 64, 2048)
                
                # Reshape back to (512, 2048)
                importance = importance_grouped.view(effective_weight.shape[0], effective_weight.shape[-1])

            elif name == "self_attn.o_proj":
                # Compute o_proj effective weight: (2048, 2048)
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                

                # Get norms from both complementary layers.
                norm_gate = get_comp_norm(comp = subset["mlp.gate_proj"])
                norm_up   = get_comp_norm(comp = subset["mlp.up_proj"])
                
                # print(f"  mlp.down_proj shape: {subset['mlp.gate_proj'].base_layer.weight.shape}")
                # print(f"  mlp.up_proj shape:   {subset['mlp.up_proj'].base_layer.weight.shape}")
                # print(f"  norm_gate shape:            {norm_gate.shape}")   
                # print(f"  norm_up shape:              {norm_up.shape}")
                # Compute importance as per:
                # importance_{ij} = |(W1+B1A1)_{ij}| * ||X_{:,i}||_2 * (norm_gate + norm_up)_j
                # importance = torch.abs(effective_weight) * X_hat * (norm_gate + norm_up)
                importance = torch.abs(effective_weight) * X_hat 
            elif name == "mlp.gate_proj":
                # Compute effective gate_proj weight: shape [8192, 2048]
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                
                comp_norm = get_comp_norm(comp = subset["mlp.down_proj"])
                # print(f"  compt_norm shape:            {comp_norm.shape}")
                # Multiply elementwise:
                # - effective_weight: shape (8192, 2048) (absolute value applied elementwise)
                # - comp_norm: shape (8192, 1), broadcast along the columns
                # - X_hat: shape (1, 2048), broadcast along the rows.
                importance = torch.abs(effective_weight) * comp_norm * X_hat  # shape: (8192, 2048)
            elif name == "mlp.up_proj":
                # effective_weight: shape [8192, 2048]
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                
                # Get the complementary weight from mlp.down_proj.
                # mlp.down_proj has shape [2048, 8192]: its columns (8192) correspond to the expanded neurons.
                comp_norm = get_comp_norm(comp = subset["mlp.down_proj"])
                # print(f"  compt_norm shape:            {comp_norm.shape}")
                
                importance = torch.abs(effective_weight) * comp_norm * X_hat
            elif name == "mlp.down_proj":
                effective_weight = weight + torch.matmul(lora_B, lora_A)
                importance = torch.abs(effective_weight) * X_hat
            else:
                print("This weight is not being recognized " + name)
                comp_weight = None

            # print(f"  importance shape: {importance.shape}")
            
            if torch.count_nonzero(score_tensor) > 0:
                # Moving average: new_score = mask_lr * current + (1 - mask_lr) * previous
                print("we use moving average")
                importance = mask_lr * importance + (1 - mask_lr) * score_tensor
            # Update the stored score.
            score_tensor.copy_(importance)


            # Rowwise Selection
            W_mask = torch.zeros_like(importance, dtype=torch.bool)
            num_to_prune = int(importance.shape[1] * prune_ratio)
            sort_res = torch.sort(importance, dim=-1, stable=True)
            indices = sort_res[1][:, :num_to_prune]
            # Scatter True into W_mask for the lowest 'num_to_prune' positions in each row
            rows = torch.arange(importance.shape[0]).unsqueeze(-1).to(importance.device)
            W_mask[rows, indices] = True
            subset[name].mask[W_mask] = 0

            # We then conduct memory recovery
            mask_tensor  = subset[name].mask
            if PBS:
                print('Perform partial brain surgery')
                # memory_recovery(mask_tensor, lora_B, lora_A, weight, reg = reg, diagonal_approx=False, inplace= True) 
                partial_brain_surgeon(mask_tensor, lora_B, lora_A, weight, reg = reg, inplace= True)

            if score_tensor.device != orig_score_device:
                score_tensor = score_tensor.to(orig_score_device)
            if subset[name].mask.device != orig_mask_device:
                subset[name].mask = subset[name].mask.to(orig_mask_device)

            #Column wise selection (worst performance)
            # W_mask = torch.zeros_like(importance, dtype=torch.bool)
            # column_importance = torch.norm(importance, p=2, dim=0)  # shape: (num_columns,)
            # num_to_prune = int(importance.shape[1] * prune_ratio)
            # sorted_col_indices = torch.argsort(column_importance, stable=True)  # shape: (num_columns,)
            # cols_to_prune = sorted_col_indices[:num_to_prune]
            # W_mask[:, cols_to_prune] = True
            # subset[name].mask[W_mask] = 0

            # Block Selection (bad performance)
            # flat = importance.view(-1)
            # num_elements = flat.numel()
            # k = int((1 - prune_ratio) * num_elements)
            # if k < 1:
            #     thresh = flat.max() + 1.0  # if k < 1, then select nothing
            # else:
            #     flat_sorted, _ = torch.sort(flat, descending=True)
            #     thresh = flat_sorted[k - 1]
            # print(f"  threshold value: {thresh}")

            # # Create the binary mask.
            # new_mask = (importance >= thresh).float()
            # print(f"  new_mask shape: {new_mask.shape}\n")
            # mask_tensor.copy_(new_mask)


        for j in range(min(128, inps.shape[0])):
            with torch.no_grad():
                pos_emb = model.base_model.model.model.rotary_emb(x = inps[j].unsqueeze(0),position_ids = position_ids)
                outs[j] = layer(
                    inps[j].unsqueeze(0),
                    attention_mask=attention_mask,
                    position_ids=position_ids,
                    position_embeddings=pos_emb
                )[0]
        inps, outs = outs, inps

    # Restore original cache setting
    model.config.use_cache = use_cache
    torch.cuda.empty_cache()

def get_comp_norm(comp):
    comp_weight = comp.base_layer.weight + torch.matmul(comp.lora_B.default.weight,
                                                        comp.lora_A.default.weight)
    # comp_weight has shape [8192, 2048]. Compute L2 norm over dim=0, yielding (1,2048),
    # then transpose to get (2048,1)
    # return torch.norm(comp_weight, p=2, dim=0, keepdim=True).transpose(0, 1).flip(dims=[0])
    # return torch.norm(comp_weight, p=2, dim=0, keepdim=True).transpose(0, 1)
    return torch.norm(comp_weight, p=2, dim=1).reshape(1, -1)


loading calibration data
dataset loading complete
pruning layer 0 name self_attn.q_proj
pruning layer 0 name self_attn.k_proj
pruning layer 0 name self_attn.v_proj
pruning layer 0 name self_attn.o_proj
pruning layer 0 name mlp.gate_proj
pruning layer 0 name mlp.up_proj
pruning layer 0 name mlp.down_proj
pruning layer 1 name self_attn.q_proj
pruning layer 1 name self_attn.k_proj
pruning layer 1 name self_attn.v_proj
pruning layer 1 name self_attn.o_proj
pruning layer 1 name mlp.gate_proj
pruning layer 1 name mlp.up_proj
pruning layer 1 name mlp.down_proj
pruning layer 2 name self_attn.q_proj
pruning layer 2 name self_attn.k_proj
pruning layer 2 name self_attn.v_proj
pruning layer 2 name self_attn.o_proj
pruning layer 2 name mlp.gate_proj
pruning layer 2 name mlp.up_proj
pruning layer 2 name mlp.down_proj
pruning layer 3 name self_attn.q_proj
pruning layer 3 name self_attn.k_proj
pruning layer 3 name self_attn.v_proj
pruning layer 3 name self_attn.o_proj
pruning layer 3 name mlp.gate_pro

Qwen3ForCausalLM(
  (model): Qwen3Model(
    (embed_tokens): Embedding(151936, 1024)
    (layers): ModuleList(
      (0-27): 28 x Qwen3DecoderLayer(
        (self_attn): Qwen3Attention(
          (q_proj): Linear(in_features=1024, out_features=2048, bias=False)
          (k_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=False)
          (o_proj): Linear(in_features=2048, out_features=1024, bias=False)
          (q_norm): Qwen3RMSNorm((128,), eps=1e-06)
          (k_norm): Qwen3RMSNorm((128,), eps=1e-06)
        )
        (mlp): Qwen3MLP(
          (gate_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (up_proj): Linear(in_features=1024, out_features=3072, bias=False)
          (down_proj): Linear(in_features=3072, out_features=1024, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen3RMSNorm((1024,), eps=1e-06)
        (post_attention_layer

In [4]:
from pruning.utils import check_sparsity

check_sparsity(model)

layer 0 sparsity 0.799805
layer 1 sparsity 0.799805
layer 2 sparsity 0.799805
layer 3 sparsity 0.799805
layer 4 sparsity 0.799805
layer 5 sparsity 0.799805
layer 6 sparsity 0.799805
layer 7 sparsity 0.799805
layer 8 sparsity 0.799805
layer 9 sparsity 0.799805
layer 10 sparsity 0.799805
layer 11 sparsity 0.799805
layer 12 sparsity 0.799805
layer 13 sparsity 0.799805
layer 14 sparsity 0.799805
layer 15 sparsity 0.799805
layer 16 sparsity 0.799805
layer 17 sparsity 0.799805
layer 18 sparsity 0.799805
layer 19 sparsity 0.799805
layer 20 sparsity 0.799805
layer 21 sparsity 0.799805
layer 22 sparsity 0.799805
layer 23 sparsity 0.799805
layer 24 sparsity 0.799805
layer 25 sparsity 0.799805
layer 26 sparsity 0.799805
layer 27 sparsity 0.799805


0.7998046875